# Flying Cars & Wright's Law

**Inflection point note (Oct 2026)**  
In 2000 a friend was asked when cars would fly. He replied:

> "Well actually, they already exist. When will we see them on the streets? I'd say as soon as they're affordable, I'm guessing the year 2075 or so, they will be commonplace."

Tesla's October 2026 Roadster demo (thruster-assisted flips / "kickflip") is a public proof-of-concept that starts the timer on that affordability-driven prediction.

This notebook models when a flying-capable car might reach a "commonplace" price (~$75k in today's dollars — expensive enough that not everyone owns one, but visible daily, like today's Rivians or higher-end Teslas).

## Equations

**Simple exponential decline** (constant % drop per year):

$$P(t) = P_0 \times (1 - r)^t$$

**Wright's Law** (cost declines with cumulative production):

$$P(n) = P_1 \times \left(\frac{n}{n_1}\right)^b, \quad b = \log_2(1 - \text{learning rate})$$

Learning rate is the fractional cost reduction per doubling of cumulative units (historically ~10–20% for autos and aerospace). Production growth maps cumulative units to calendar time.

## Inflation adjustment (nominal vs real)

Wright’s Law and the simple exponential model are best interpreted as **real** (inflation-adjusted) cost declines — technology getting cheaper relative to the general price level.

To see what you would actually pay in future dollars, we apply a vehicle/EV inflation rate on top.

Historical context (BLS CPI – New Vehicles):
- Long-term average ~1–2% per year (quality-adjusted).
- 2000–2025: ~0.9% annualized.
- Big spike 2020–2022, then flattening.

Default in the model: **2%** annual vehicle inflation. You can change it.


In [1]:
import math

def years_to_price(
    start_price=300_000,
    target=75_000,
    start_year=2028,
    # Wright's Law params (used when use_wright=True)
    learning_rate=0.15,
    initial_annual_units=1_000,
    annual_growth=0.50,
    # Simple exponential params (used when use_wright=False)
    annual_decline=0.07,
    use_wright=True,
    # Inflation (applied to get nominal future prices)
    vehicle_inflation=0.02,
    max_years=100,
    return_prices=False
):
    """
    Forecast years until real price reaches target.
    Also returns nominal (future dollars) and real (today's dollars) prices
    if return_prices=True.
    """
    if start_price <= target:
        return (start_year, start_price, start_price) if return_prices else start_year

    if not use_wright:
        years = math.log(target / start_price) / math.log(1 - annual_decline)
        year = start_year + years
        real_price = target
        nominal_price = target * (1 + vehicle_inflation) ** years
        return (year, nominal_price, real_price) if return_prices else year

    # Wright's Law (real prices)
    b = math.log2(1 - learning_rate)
    cumulative = 0.0
    year = start_year
    annual_units = float(initial_annual_units)

    for _ in range(max_years):
        cumulative += annual_units
        real_price = start_price * (cumulative / initial_annual_units) ** b
        if real_price <= target:
            years_elapsed = year - start_year
            nominal_price = real_price * (1 + vehicle_inflation) ** years_elapsed
            return (year, nominal_price, real_price) if return_prices else year
        annual_units *= (1 + annual_growth)
        year += 1

    return None


def observed_learning_rate(price1, cumul1, price2, cumul2):
    """Back out learning rate from two observed (price, cumulative units) points."""
    if cumul2 <= cumul1 or price2 >= price1:
        return None
    b = math.log(price2 / price1) / math.log(cumul2 / cumul1)
    return 1 - 2 ** b


## Example runs

Default assumptions: $300k starting price in 2028, $75k target.

In [2]:
print("=== With 2% vehicle inflation ===\n")

for label, kwargs in [
    ("Wright's Law (15% LR, 50% growth)", dict(use_wright=True)),
    ("Wright's Law (20% LR, 80% growth)", dict(use_wright=True, learning_rate=0.20, annual_growth=0.80)),
    ("Simple exponential (7%)", dict(use_wright=False)),
    ("Simple exponential (5%)", dict(use_wright=False, annual_decline=0.05)),
    ("Simple exponential (10%)", dict(use_wright=False, annual_decline=0.10)),
]:
    year, nominal, real = years_to_price(return_prices=True, **kwargs)
    print(f"{label}:")
    print(f"  Year: {year:.1f}")
    print(f"  Nominal price (future $): ${nominal:,.0f}")
    print(f"  Real price (today's $):   ${real:,.0f}\n")


=== With 2% vehicle inflation ===

Wright's Law (15% LR, 50% growth):
  Year: 2040.0
  Nominal price (future $): $95,000
  Real price (today's $):   $75,000

Wright's Law (20% LR, 80% growth):
  Year: 2034.0
  Nominal price (future $): $84,000
  Real price (today's $):   $75,000

Simple exponential (7%):
  Year: 2047.1
  Nominal price (future $): $109,000
  Real price (today's $):   $75,000

Simple exponential (5%):
  Year: 2055.0
  Nominal price (future $): $127,000
  Real price (today's $):   $75,000

Simple exponential (10%):
  Year: 2041.2
  Nominal price (future $): $97,000
  Real price (today's $):   $75,000



## Updating with real data

As production numbers and prices become public, replace the guesses:

```python
params = dict(
    start_price=275_000,          # actual early transaction price
    learning_rate=observed_learning_rate(300_000, 500, 240_000, 2_000),
    annual_growth=0.35,           # recent observed growth
    target=75_000,
    start_year=2028,
)
print(years_to_price(**params))
```

15% is a reasonable prior (historical auto ~10–15%, aerospace often 15–20%). Early Rocket versions will learn more slowly due to low volume and certification constraints.

## Caveats

- Price is necessary but not sufficient. Regulatory approval, insurance, infrastructure, and thruster energy costs also matter.
- The current Roadster Rocket is invite-only and not approved for public roads; 2028 is a production-start target.
- "Commonplace" is defined here as a price where the vehicle is regularly visible (analogous to today's Rivians). You can change `target` to a multiple of average new-car transaction price if you prefer a relative definition.
- Single-point estimates look more precise than they are. Run a small grid of learning rates and growth rates for a range.